# Esquema: regresión + sklearn + PyTorch (MVP)

CSV con faltantes → **tratamiento manual** en pandas → split → varios modelos **sklearn** en `Pipeline` → **MLP en PyTorch** → comparación en test.

| Paso | Contenido |
|------|-----------|
| 1–5 | CSV (`data/datos_casas.csv`), tipos, imputación, dummies, split |
| 6 | Entrenar modelos **sklearn** (`build_models()`) |
| 7 | Entrenar red **PyTorch** (`HousePriceNet`) |
| 8 | **Comparación global** sklearn + PyTorch en test |

> Ejecuta el notebook desde `13-esquemas-sklearn-pytorch/`.


## 1. Importar CSV y revisar datos

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

df = pd.read_csv("data/datos_casas.csv")
print("Tipos:\n", df.dtypes)
print("\nFaltantes:\n", df.isna().sum())
display(df)


Tipos:
 Metros_Cuadrados    float64
Zona                 object
Precio                int64
dtype: object

Faltantes:
 Metros_Cuadrados    1
Zona                1
Precio              0
dtype: int64


,Metros_Cuadrados,Zona,Precio
0,45.0,Centro,125000
1,52.0,Periferia,138000
2,NaN,Centro,132000
3,61.0,Periferia,155000
4,70.0,Centro,172000
5,85.0,NaN,195000
6,95.0,Periferia,210000
7,110.0,Centro,245000
8,130.0,Periferia,285000
9,150.0,Centro,320000


## 2. Target numérico (`Precio`)

In [2]:
df = df.dropna(subset=["Precio"]).copy()
y = df["Precio"]


## 3–4. Features (manual)

In [3]:
metros = df["Metros_Cuadrados"].astype(float)
X_num = pd.DataFrame({"Metros_Cuadrados": metros.fillna(metros.median())})
zona = df["Zona"].fillna("Desconocida").astype(str)
X_cat = pd.get_dummies(zona, prefix="Zona", dtype=float)
X = pd.concat([X_num, X_cat], axis=1)


## 5. Split

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


## 6. Modelos sklearn en Pipeline

In [5]:
def build_models():
    """Comenta entradas del dict para excluir modelos."""
    from sklearn.ensemble import (
        GradientBoostingRegressor,
        HistGradientBoostingRegressor,
        RandomForestRegressor,
    )
    from sklearn.linear_model import Lasso, LinearRegression, Ridge
    from sklearn.neighbors import KNeighborsRegressor
    from sklearn.tree import DecisionTreeRegressor
    from xgboost import XGBRegressor
    from catboost import CatBoostRegressor

    return {
        "LinearRegression": LinearRegression(),
        "Ridge": Ridge(random_state=RANDOM_STATE),
        "Lasso": Lasso(random_state=RANDOM_STATE, max_iter=5000),
        "DecisionTree": DecisionTreeRegressor(
            criterion="squared_error",
            splitter="best",
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            min_weight_fraction_leaf=0.0,
            max_features=None,
            max_leaf_nodes=None,
            min_impurity_decrease=0.0,
            random_state=RANDOM_STATE,
        ),
        "RandomForest": RandomForestRegressor(
            n_estimators=100,
            criterion="squared_error",
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            min_weight_fraction_leaf=0.0,
            max_features=1.0,
            max_leaf_nodes=None,
            min_impurity_decrease=0.0,
            bootstrap=True,
            oob_score=False,
            max_samples=None,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "GradientBoosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
        "HistGradientBoosting": HistGradientBoostingRegressor(random_state=RANDOM_STATE),
        "KNN": KNeighborsRegressor(n_neighbors=5, n_jobs=-1),
        "XGBoost": XGBRegressor(
            random_state=RANDOM_STATE, verbosity=0, n_estimators=100, n_jobs=-1
        ),
        "CatBoost": CatBoostRegressor(
            random_state=RANDOM_STATE,
            verbose=False,
            iterations=100,
            allow_writing_files=False,
        ),
    }


RANDOM_STATE = 42
MODELS = build_models()

filas = []
predicciones_test = {}
for nombre, modelo in MODELS.items():
    pipe = make_pipeline(StandardScaler(), modelo)
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    predicciones_test[nombre] = pred
    filas.append({
        "modelo": nombre,
        "R2": r2_score(y_test, pred),
        "MSE": mean_squared_error(y_test, pred),
    })
# métricas agregadas en el paso 8



## 7. Red neuronal (PyTorch)

Misma **X** / **y** que sklearn. Escalado solo en train. **MLP tabular**: capas densas + `BatchNorm1d` + ReLU + `Dropout` + salida lineal.

> Referencia: [12-pytorch/00-pytorch-cheat-sheet.ipynb](../12-pytorch/00-pytorch-cheat-sheet.ipynb)


In [6]:
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)

# Escalado para la red (aprendido solo en train)
scaler_nn = StandardScaler()
X_tr = scaler_nn.fit_transform(X_train)
X_te = scaler_nn.transform(X_test)

n_in = X_tr.shape[1]


# MLP para regresión: capas densas + BatchNorm + ReLU + Dropout; salida lineal (un solo valor).
class HousePriceNet(nn.Module):
    def __init__(self, n_features: int, dropout_rate: float = 0.3):
        super().__init__()
        # Bloques de ancho decreciente: patrón habitual en tabular (ancho → estrecho).
        self.network = nn.Sequential(
            nn.Linear(n_features, 256),
            nn.BatchNorm1d(256),  # Normaliza activaciones por batch.
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate * 0.5),  # Menos dropout cerca de la salida.
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, 1),  # Sin sigmoid: valor continuo (objetivo escalado).
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)  # x: (batch, n_features) → (batch, 1)


modelo_nn = HousePriceNet(n_in).to(device)
criterio = nn.MSELoss()
optimizador = torch.optim.Adam(modelo_nn.parameters(), lr=1e-2)

# Escalar también y (solo stats de train) — estabiliza el MLP con pocos datos
y_mean, y_std = float(y_train.mean()), float(y_train.std()) + 1e-8
y_tr_norm = ((y_train - y_mean) / y_std).values.reshape(-1, 1)

X_t = torch.tensor(X_tr, dtype=torch.float32, device=device)
y_t = torch.tensor(y_tr_norm, dtype=torch.float32, device=device)

EPOCHS = 400
for ep in range(EPOCHS):
    modelo_nn.train()
    optimizador.zero_grad()
    perdida = criterio(modelo_nn(X_t), y_t)
    perdida.backward()
    optimizador.step()

modelo_nn.eval()
with torch.no_grad():
    pred_norm = modelo_nn(torch.tensor(X_te, dtype=torch.float32, device=device))
    pred_nn = pred_norm.cpu().numpy().ravel() * y_std + y_mean

mse_nn = mean_squared_error(y_test, pred_nn)
r2_nn = r2_score(y_test, pred_nn)

predicciones_test["PyTorch_MLP"] = pred_nn



## 8. Comparación de todos los modelos (sklearn + PyTorch)

Misma partición **test** para todos. Tabla ordenada por **R²** (mayor = mejor).


In [7]:
comparacion = pd.concat(
    [
        pd.DataFrame(filas),
        pd.DataFrame([{"modelo": "PyTorch_MLP", "R2": r2_nn, "MSE": mse_nn}]),
    ],
    ignore_index=True,
).sort_values("R2", ascending=False)

display(comparacion.round(4))

mejor = comparacion.iloc[0]
print(f"\nMejor modelo en test: {mejor['modelo']}")
print(f"  R² = {mejor['R2']:.4f} | MSE = {mejor['MSE']:,.0f}")



,modelo,R2,MSE
2,Lasso,0.9334,4.331365e+08
0,LinearRegression,0.9298,4.564532e+08
1,Ridge,0.9204,5.179841e+08
8,XGBoost,0.9015,6.410019e+08
4,RandomForest,0.8580,9.240190e+08
10,PyTorch_MLP,0.7889,1.373364e+09
3,DecisionTree,0.7252,1.787667e+09
5,GradientBoosting,0.5957,2.630185e+09
7,KNN,0.5905,2.664080e+09
9,CatBoost,0.4504,3.575504e+09



Mejor modelo en test: Lasso
  R² = 0.9334 | MSE = 433,136,541
